# UrbanFloodBench — Full Dataset Preprocessing

**Dataset:** Full 18 GB dataset — `Models/Model_{id}/train/event_{id}/` structure.
**Models:** Model 1 (68 train events, 29 test) + Model 2 (69 train events, 30 test).
**Key decisions:**
- Norm stats computed from **training data only** (no leakage)
- Target = depth above invert/min_elevation → predict delta per step
- Events streamed one-at-a-time to avoid OOM
- Outputs: per-model bundles in `/kaggle/working/preprocessed/model_{id}/`

## Cell 1 — Install dependencies

In [4]:
import subprocess, sys, torch

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

print(f'PyTorch version: {torch.__version__}')

# torch_geometric is all we need — no optional extensions required
pip('torch_geometric')
pip('lightgbm', 'pyarrow', 'fastparquet', 'tqdm')

print('Done.')

PyTorch version: 2.9.0+cpu
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 17.9 MB/s eta 0:00:00
Done.


## Cell 2 — Imports & paths

In [5]:
import os, gc, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

# ── Paths ──────────────────────────────────────────────────────────────────
# Update DATA_ROOT to your Kaggle dataset slug for the full 18GB dataset
DATA_ROOT   = Path('/kaggle/input/datasets/jiayulim/urbanfloodbench')
MODELS_ROOT = DATA_ROOT / 'Models'
OUT_ROOT    = Path('/kaggle/working/preprocessed')
OUT_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_IDS = [1, 2]

for mid in MODEL_IDS:
    p = MODELS_ROOT / f'Model_{mid}'
    assert p.exists(), f'Missing: {p}'
    train_events = sorted([e for e in (p / 'train').iterdir()
                           if e.is_dir() and e.name.startswith('event_')])
    test_events  = sorted([e for e in (p / 'test').iterdir()
                           if e.is_dir() and e.name.startswith('event_')])
    print(f'Model {mid}: {len(train_events)} train events, {len(test_events)} test events')

print('Data root verified.')

Model 1: 68 train events, 29 test events
Model 2: 69 train events, 30 test events
Data root verified.


## Cell 3 — Helper functions

In [6]:
def load_csv(path):
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
    return df

def find_col(df, *keywords):
    for kw in keywords:
        hits = [c for c in df.columns if kw in c]
        if hits:
            return hits[0]
    raise ValueError(f'No column matching {keywords} in {list(df.columns)}')

def ensure_node_col(df, preferred='node_idx'):
    if preferred in df.columns:
        return df
    cands = [c for c in df.columns if ('node' in c and ('idx' in c or 'id' in c))]
    if cands:
        return df.rename(columns={cands[0]: preferred})
    return df.rename(columns={df.columns[0]: preferred})

def ensure_edge_col(df, preferred='edge_idx'):
    if preferred in df.columns:
        return df
    cands = [c for c in df.columns if ('edge' in c and ('idx' in c or 'id' in c))]
    if cands:
        return df.rename(columns={cands[0]: preferred})
    return df.rename(columns={df.columns[0]: preferred})

def ensure_timestep_col(df):
    if 'timestep' in df.columns:
        return df
    cands = [c for c in df.columns if 'time' in c or c == 't']
    if cands:
        return df.rename(columns={cands[0]: 'timestep'})
    return df.rename(columns={df.columns[0]: 'timestep'})

def compute_stats(df, cols, clip_sigma=4.0):
    stats = {}
    for c in cols:
        if c not in df.columns:
            continue
        vals = df[c].dropna().astype(float)
        mu, sig = float(vals.mean()), float(vals.std())
        if sig < 1e-8:
            stats[c] = {'mean': mu, 'std': 1.0, 'constant': True}
        else:
            stats[c] = {'mean': mu, 'std': sig, 'constant': False,
                        'clip_lo': mu - clip_sigma * sig,
                        'clip_hi': mu + clip_sigma * sig}
    return stats

def normalise_df(df, stats, cols):
    out = df.copy()
    for c in cols:
        if c not in stats or c not in out.columns:
            continue
        s = stats[c]
        vals = out[c].astype(float)
        if 'clip_lo' in s:
            vals = vals.clip(s['clip_lo'], s['clip_hi'])
        out[f'{c}_norm'] = (vals - s['mean']) / (s['std'] + 1e-8)
    return out

def get_event_dirs(model_dir, split):
    split_dir = model_dir / split
    return sorted(
        [d for d in split_dir.iterdir() if d.is_dir() and d.name.startswith('event_')],
        key=lambda d: int(d.name.split('_')[1])
    )

def load_event_dynamic(event_dir):
    dyn_1d = ensure_timestep_col(ensure_node_col(
        load_csv(event_dir / '1d_nodes_dynamic_all.csv')))
    dyn_2d = ensure_timestep_col(ensure_node_col(
        load_csv(event_dir / '2d_nodes_dynamic_all.csv')))
    dyn_1d.sort_values(['node_idx', 'timestep'], inplace=True)
    dyn_2d.sort_values(['node_idx', 'timestep'], inplace=True)
    return dyn_1d, dyn_2d

print('Helpers defined.')

Helpers defined.


## Cell 4 — Load static files

In [7]:
def load_static(model_dir):
    train_dir = model_dir / 'train'

    n1d = ensure_node_col(load_csv(train_dir / '1d_nodes_static.csv'))

    n2d_path = train_dir / '2d_nodes_static.csv'
    if not n2d_path.exists():
        n2d_path = train_dir / '2d_nodes_index.csv'
    n2d = ensure_node_col(load_csv(n2d_path))

    e1d = ensure_edge_col(load_csv(train_dir / '1d_edges_static.csv'))

    e2d_path = train_dir / '2d_edges_static.csv'
    if not e2d_path.exists():
        e2d_path = train_dir / '2d_edges_index.csv'
    e2d = ensure_edge_col(load_csv(e2d_path))

    ei1d = load_csv(train_dir / '1d_edge_index.csv')
    ei2d = load_csv(train_dir / '2d_edge_index.csv')
    conn = load_csv(train_dir / '1d2d_connections.csv')

    n1d_col = find_col(conn, 'node_1d', '1d_node', 'node1d')
    n2d_col = find_col(conn, 'node_2d', '2d_node', 'node2d')
    conn = conn.rename(columns={n1d_col: 'node_1d', n2d_col: 'node_2d'})

    return n1d, n2d, e1d, e2d, ei1d, ei2d, conn

# Quick check
for mid in MODEL_IDS:
    n1d, n2d, e1d, e2d, ei1d, ei2d, conn = load_static(MODELS_ROOT / f'Model_{mid}')
    print(f'Model {mid}: 1D nodes={len(n1d)}, 2D nodes={len(n2d)}, '
          f'1D edges={len(e1d)}, 2D edges={len(e2d)}, '
          f'1D-2D links={len(conn)}')

Model 1: 1D nodes=17, 2D nodes=3716, 1D edges=16, 2D edges=7935, 1D-2D links=16
Model 2: 1D nodes=198, 2D nodes=4299, 1D edges=197, 2D edges=9876, 1D-2D links=197


## Cell 5 — Static feature engineering

In [8]:
def engineer_static_features(n1d, n2d, e1d, e2d, ei1d, ei2d, conn):
    n1d, n2d, e1d, e2d = n1d.copy(), n2d.copy(), e1d.copy(), e2d.copy()

    cols = {}
    cols['px_1d']    = find_col(n1d, 'position_x', '1d_position_x', 'x')
    cols['py_1d']    = find_col(n1d, 'position_y', '1d_position_y', 'y')
    cols['depth_1d'] = find_col(n1d, 'depth')
    cols['inv_el']   = find_col(n1d, 'invert_elevation', 'invert')
    cols['surf_el']  = find_col(n1d, 'surface_elevation', 'surface')
    cols['base_area']= find_col(n1d, 'base_area', 'area')

    cols['px_2d']   = find_col(n2d, 'position_x', '2d_position_x', 'x')
    cols['py_2d']   = find_col(n2d, 'position_y', '2d_position_y', 'y')
    cols['area_2d'] = find_col(n2d, 'area')
    cols['rough_2d']= find_col(n2d, 'roughness')
    cols['min_el']  = find_col(n2d, 'min_elevation', 'minimum_elevation')
    cols['elev_2d'] = find_col(n2d, 'centroid_elevation', 'elevation')
    cols['aspect']  = find_col(n2d, 'aspect')
    cols['curv']    = find_col(n2d, 'curvature')
    cols['flow_acc']= find_col(n2d, 'flow_accumulation', 'flow_acc')

    cols['from_1d'] = find_col(ei1d, 'from_node', 'from')
    cols['to_1d']   = find_col(ei1d, 'to_node', 'to')
    cols['from_2d'] = find_col(ei2d, 'from_node', 'from')
    cols['to_2d']   = find_col(ei2d, 'to_node', 'to')

    cols['len_1d']  = find_col(e1d, 'length', '1d_edge_length')
    cols['slope_1d']= find_col(e1d, 'slope')
    cols['diam_1d'] = find_col(e1d, 'diameter')
    cols['rpx_1d']  = find_col(e1d, 'relative_position_x')
    cols['rpy_1d']  = find_col(e1d, 'relative_position_y')

    cols['len_2d']  = find_col(e2d, '2d_length', 'length')
    cols['slope_2d']= find_col(e2d, 'slope')
    cols['face_2d'] = find_col(e2d, 'face_length', 'face')
    cols['rpx_2d']  = find_col(e2d, 'relative_position_x')
    cols['rpy_2d']  = find_col(e2d, 'relative_position_y')

    # Drop constant 1D edge cols (shape, roughness)
    for kw in ['shape', 'roughness']:
        for c in [c for c in e1d.columns if kw in c]:
            if e1d[c].nunique() <= 2:
                e1d.drop(columns=c, inplace=True, errors='ignore')

    # 1D node derived features
    n1d['fill_ratio'] = (
        (n1d[cols['surf_el']] - n1d[cols['inv_el']]) /
        n1d[cols['depth_1d']].replace(0, np.nan)
    ).fillna(1.0).clip(0, 5)

    all_edge_nodes = np.concatenate([
        ei1d[cols['from_1d']].values, ei1d[cols['to_1d']].values
    ])
    deg_map = pd.Series(all_edge_nodes).value_counts().to_dict()
    n1d['degree_1d'] = n1d['node_idx'].map(deg_map).fillna(0).astype(int)

    n2d_conn_count = conn.groupby('node_1d').size().to_dict()
    n1d['n_2d_connections'] = n1d['node_idx'].map(n2d_conn_count).fillna(0).astype(int)

    area_lookup = n2d.set_index('node_idx')[cols['area_2d']].to_dict()
    conn_area = conn.copy()
    conn_area['area_2d_val'] = conn_area['node_2d'].map(area_lookup).fillna(0)
    catchment_map = conn_area.groupby('node_1d')['area_2d_val'].sum().to_dict()
    n1d['catchment_area'] = n1d['node_idx'].map(catchment_map).fillna(0)

    # 2D node derived features
    asp_rad = np.deg2rad(n2d[cols['aspect']])
    n2d['aspect_sin'] = np.sin(asp_rad)
    n2d['aspect_cos'] = np.cos(asp_rad)
    n2d['area_log']   = np.log1p(n2d[cols['area_2d']])
    n2d['flow_acc_log'] = np.log1p(n2d[cols['flow_acc']])
    n2d['terrain_slope_proxy'] = (
        (n2d[cols['elev_2d']] - n2d[cols['min_el']]).clip(lower=0) /
        (np.sqrt(n2d[cols['area_2d']]) + 1e-6)
    )
    n2d['fill_capacity'] = (
        (n2d[cols['elev_2d']] - n2d[cols['min_el']]).clip(lower=0) * n2d[cols['area_2d']]
    )
    n2d['catchment_potential'] = n2d['flow_acc_log'] * n2d[cols['area_2d']]

    coupled_2d = set(conn['node_2d'].values)
    n2d['has_1d_connection'] = n2d['node_idx'].isin(coupled_2d).astype(float)

    pos_1d_df = n1d.set_index('node_idx')[[cols['px_1d'], cols['py_1d']]]
    cp = conn[['node_1d', 'node_2d']].merge(
        pos_1d_df.rename(columns={cols['px_1d']: 'x1d', cols['py_1d']: 'y1d'}),
        left_on='node_1d', right_index=True, how='left'
    )
    pos_2d_df = n2d.set_index('node_idx')[[cols['px_2d'], cols['py_2d']]]
    cp = cp.merge(
        pos_2d_df.rename(columns={cols['px_2d']: 'x2d', cols['py_2d']: 'y2d'}),
        left_on='node_2d', right_index=True, how='left'
    )
    cp['dist'] = np.sqrt((cp['x1d'] - cp['x2d'])**2 + (cp['y1d'] - cp['y2d'])**2)
    dist_map = cp.groupby('node_2d')['dist'].min().to_dict()
    max_dist = max(dist_map.values()) if dist_map else 1.0
    n2d['dist_to_1d'] = n2d['node_idx'].map(dist_map).fillna(max_dist)

    depth_1d_lookup = n1d.set_index('node_idx')[cols['depth_1d']].to_dict()
    conn_d = conn.copy()
    conn_d['depth_val'] = conn_d['node_1d'].map(depth_1d_lookup)
    avg_depth_map = conn_d.groupby('node_2d')['depth_val'].mean().to_dict()
    n2d['connected_1d_depth'] = n2d['node_idx'].map(avg_depth_map).fillna(0.0)

    # 1D edge derived features
    e1d['head_loss'] = e1d[cols['slope_1d']] * e1d[cols['len_1d']]
    e1d['capacity_proxy'] = (
        e1d[cols['diam_1d']].clip(lower=1e-3) ** (8/3) *
        np.abs(e1d[cols['slope_1d']]).clip(lower=1e-6) ** 0.5
    )

    # 2D edge derived features
    e2d['unit_conveyance'] = (
        e2d[cols['face_2d']] * np.abs(e2d[cols['slope_2d']]).clip(lower=1e-6) ** 0.5
    )
    e2d['head_loss_2d'] = e2d[cols['slope_2d']] * e2d[cols['len_2d']]
    lens = np.sqrt(e2d[cols['rpx_2d']]**2 + e2d[cols['rpy_2d']]**2).replace(0, 1e-6)
    e2d['dx_norm'] = e2d[cols['rpx_2d']] / lens
    e2d['dy_norm'] = e2d[cols['rpy_2d']] / lens

    return n1d, n2d, e1d, e2d, cols

print('engineer_static_features() defined.')

engineer_static_features() defined.


## Cell 6 — Collect normalisation stats (streaming all train events)

In [9]:
def collect_train_stats(model_dir, n1d, n2d, cols):
    invert_lookup  = n1d.set_index('node_idx')[cols['inv_el']].to_dict()
    minelev_lookup = n2d.set_index('node_idx')[cols['min_el']].to_dict()

    class OnlineStats:
        def __init__(self):
            self.n, self.mean, self.M2 = 0, 0.0, 0.0
        def update(self, vals):
            for x in vals:
                if np.isnan(x) or np.isinf(x):
                    continue
                self.n += 1
                delta = x - self.mean
                self.mean += delta / self.n
                self.M2 += delta * (x - self.mean)
        def std(self):
            return float(np.sqrt(self.M2 / max(self.n - 1, 1)))

    acc = {k: OnlineStats() for k in
           ['wl_1d', 'wl_2d', 'depth_1d', 'depth_2d', 'delta_1d', 'delta_2d', 'rain']}

    train_events = get_event_dirs(model_dir, 'train')
    wl_1d_col = wl_2d_col = rain_col = None

    for ev_dir in tqdm(train_events, desc=f'  Scanning {model_dir.name}'):
        dyn_1d, dyn_2d = load_event_dynamic(ev_dir)

        if wl_1d_col is None:
            wl_1d_col = find_col(dyn_1d, 'water_level', 'wl')
            wl_2d_col = find_col(dyn_2d, 'water_level', 'wl')
            rain_col  = find_col(dyn_2d, 'rainfall', 'rain')

        acc['wl_1d'].update(dyn_1d[wl_1d_col].dropna().values)
        acc['wl_2d'].update(dyn_2d[wl_2d_col].dropna().values)

        dyn_1d['_inv'] = dyn_1d['node_idx'].map(invert_lookup)
        dyn_2d['_min'] = dyn_2d['node_idx'].map(minelev_lookup)
        acc['depth_1d'].update((dyn_1d[wl_1d_col] - dyn_1d['_inv']).clip(0).dropna().values)
        acc['depth_2d'].update((dyn_2d[wl_2d_col] - dyn_2d['_min']).clip(0).dropna().values)

        for nid, grp in dyn_1d.groupby('node_idx'):
            d = (grp.sort_values('timestep')[wl_1d_col] - invert_lookup.get(nid, 0)).clip(0)
            acc['delta_1d'].update(d.diff().dropna().values)
        for nid, grp in dyn_2d.groupby('node_idx'):
            d = (grp.sort_values('timestep')[wl_2d_col] - minelev_lookup.get(nid, 0)).clip(0)
            acc['delta_2d'].update(d.diff().dropna().values)

        acc['rain'].update(dyn_2d[rain_col].dropna().values)
        del dyn_1d, dyn_2d
        gc.collect()

    stats = {
        'wl_1d_col': wl_1d_col, 'wl_2d_col': wl_2d_col, 'rain_col': rain_col,
        'wl_1d':    {'mean': acc['wl_1d'].mean,    'std': acc['wl_1d'].std()},
        'wl_2d':    {'mean': acc['wl_2d'].mean,    'std': acc['wl_2d'].std()},
        'depth_1d': {'mean': acc['depth_1d'].mean, 'std': acc['depth_1d'].std()},
        'depth_2d': {'mean': acc['depth_2d'].mean, 'std': acc['depth_2d'].std()},
        'delta_1d': {'mean': 0.0,                  'std': acc['delta_1d'].std()},
        'delta_2d': {'mean': 0.0,                  'std': acc['delta_2d'].std()},
        'rain':     {'mean': acc['rain'].mean,      'std': acc['rain'].std()},
    }
    print(f'  WL std   -> 1D: {stats["wl_1d"]["std"]:.4f} ft,  2D: {stats["wl_2d"]["std"]:.4f} ft')
    print(f'  Depth std -> 1D: {stats["depth_1d"]["std"]:.4f} ft,  2D: {stats["depth_2d"]["std"]:.4f} ft')
    print(f'  Delta std -> 1D: {stats["delta_1d"]["std"]:.5f} ft,  2D: {stats["delta_2d"]["std"]:.5f} ft')
    return stats

print('collect_train_stats() defined.')

collect_train_stats() defined.


## Cell 7 — Build dynamic tensors for one event

In [10]:
def build_event_tensors(event_dir, n1d, n2d, conn, cols, stats,
                        n_lag=3, rain_windows=(6, 12, 24)):
    dyn_1d, dyn_2d = load_event_dynamic(event_dir)

    wl_1d_col = stats['wl_1d_col']
    wl_2d_col = stats['wl_2d_col']
    rain_col  = stats['rain_col']

    invert_lookup  = n1d.set_index('node_idx')[cols['inv_el']].to_dict()
    minelev_lookup = n2d.set_index('node_idx')[cols['min_el']].to_dict()
    N_1D = len(n1d)
    N_2D = len(n2d)

    # Depth transformation
    dyn_1d['_inv']  = dyn_1d['node_idx'].map(invert_lookup)
    dyn_1d['depth'] = (dyn_1d[wl_1d_col] - dyn_1d['_inv']).clip(lower=0.0)
    dyn_2d['_min']  = dyn_2d['node_idx'].map(minelev_lookup)
    dyn_2d['depth'] = (dyn_2d[wl_2d_col] - dyn_2d['_min']).clip(lower=0.0)

    # Lag + delta
    def add_lag_delta(df, depth_col):
        df = df.sort_values(['node_idx', 'timestep']).copy()
        grp = df.groupby('node_idx')[depth_col]
        for lag in range(1, n_lag + 1):
            df[f'lag{lag}'] = grp.shift(lag).fillna(0.0)
        df['delta'] = df[depth_col] - df['lag1']
        return df

    dyn_1d = add_lag_delta(dyn_1d, 'depth')
    dyn_2d = add_lag_delta(dyn_2d, 'depth')

    # Rainfall features on 2D nodes
    dyn_2d = dyn_2d.sort_values(['node_idx', 'timestep']).copy()
    rain_grp = dyn_2d.groupby('node_idx')[rain_col]
    for w in rain_windows:
        dyn_2d[f'cumrain_{w}'] = rain_grp.transform(
            lambda x: x.rolling(w, min_periods=1).sum()
        )
    dyn_2d['rain_delta'] = rain_grp.diff().fillna(0.0)

    # Broadcast rainfall to 1D nodes
    rain_cols_list = [rain_col] + [f'cumrain_{w}' for w in rain_windows] + ['rain_delta']
    rain_2d_ts = dyn_2d[['timestep', 'node_idx'] + rain_cols_list].copy()
    merged = conn[['node_1d', 'node_2d']].merge(
        rain_2d_ts.rename(columns={'node_idx': 'node_2d'}), on='node_2d', how='left'
    )
    rain_1d = merged.groupby(['node_1d', 'timestep'])[rain_cols_list].mean().reset_index()
    rain_1d = rain_1d.rename(columns={'node_1d': 'node_idx'})
    dyn_1d = dyn_1d.merge(rain_1d, on=['node_idx', 'timestep'], how='left')
    for c in rain_cols_list:
        dyn_1d[c] = dyn_1d[c].fillna(0.0)

    # Normalise
    d1_mean, d1_std  = stats['depth_1d']['mean'], stats['depth_1d']['std'] + 1e-8
    d2_mean, d2_std  = stats['depth_2d']['mean'], stats['depth_2d']['std'] + 1e-8
    dt1_std = stats['delta_1d']['std'] + 1e-8
    dt2_std = stats['delta_2d']['std'] + 1e-8
    r_mean,  r_std   = stats['rain']['mean'],     stats['rain']['std']  + 1e-8

    dyn_1d['depth_norm'] = (dyn_1d['depth'] - d1_mean) / d1_std
    dyn_2d['depth_norm'] = (dyn_2d['depth'] - d2_mean) / d2_std
    for lag in range(1, n_lag + 1):
        dyn_1d[f'lag{lag}_norm'] = (dyn_1d[f'lag{lag}'] - d1_mean) / d1_std
        dyn_2d[f'lag{lag}_norm'] = (dyn_2d[f'lag{lag}'] - d2_mean) / d2_std
    dyn_1d['delta_norm'] = dyn_1d['delta'] / dt1_std
    dyn_2d['delta_norm'] = dyn_2d['delta'] / dt2_std
    for c in rain_cols_list:
        dyn_1d[f'{c}_norm'] = (dyn_1d[c] - r_mean) / r_std
        dyn_2d[f'{c}_norm'] = (dyn_2d[c] - r_mean) / r_std

    # Feature column lists
    lag_cols  = [f'lag{i}_norm' for i in range(1, n_lag + 1)]
    rain_norm = [f'{c}_norm' for c in rain_cols_list]
    DYN_1D = ['depth_norm'] + lag_cols + [c for c in rain_norm if c in dyn_1d.columns]
    DYN_2D = ['depth_norm'] + lag_cols + [c for c in rain_norm if c in dyn_2d.columns]

    # Pivot to (T, N, F)
    timesteps = sorted(dyn_1d['timestep'].unique())
    T = len(timesteps)
    t_map    = {t: i for i, t in enumerate(timesteps)}
    n1d_map  = {nid: i for i, nid in enumerate(sorted(n1d['node_idx'].unique()))}
    n2d_map  = {nid: i for i, nid in enumerate(sorted(n2d['node_idx'].unique()))}

    x_1d     = np.zeros((T, N_1D, len(DYN_1D)), dtype=np.float32)
    x_2d     = np.zeros((T, N_2D, len(DYN_2D)), dtype=np.float32)
    delta_1d = np.zeros((T, N_1D), dtype=np.float32)
    delta_2d = np.zeros((T, N_2D), dtype=np.float32)
    depth_1d = np.zeros((T, N_1D), dtype=np.float32)
    depth_2d = np.zeros((T, N_2D), dtype=np.float32)

    for ts, grp in dyn_1d.groupby('timestep'):
        ti = t_map[ts]
        ni = grp['node_idx'].map(n1d_map).dropna().astype(int)
        valid = ni.index
        x_1d[ti, ni.values, :]  = grp.loc[valid, DYN_1D].values
        delta_1d[ti, ni.values] = grp.loc[valid, 'delta_norm'].values
        depth_1d[ti, ni.values] = grp.loc[valid, 'depth'].values

    for ts, grp in dyn_2d.groupby('timestep'):
        ti = t_map[ts]
        ni = grp['node_idx'].map(n2d_map).dropna().astype(int)
        valid = ni.index
        x_2d[ti, ni.values, :]  = grp.loc[valid, DYN_2D].values
        delta_2d[ti, ni.values] = grp.loc[valid, 'delta_norm'].values
        depth_2d[ti, ni.values] = grp.loc[valid, 'depth'].values

    # Time encoding
    t_arr = np.arange(T, dtype=np.float32) / max(T - 1, 1)
    time_enc = np.stack([t_arr, np.sin(2*np.pi*t_arr), np.cos(2*np.pi*t_arr)], axis=1)

    result = {
        'x_1d':      torch.from_numpy(x_1d),
        'x_2d':      torch.from_numpy(x_2d),
        'delta_1d':  torch.from_numpy(delta_1d),
        'delta_2d':  torch.from_numpy(delta_2d),
        'depth_1d':  torch.from_numpy(depth_1d),
        'depth_2d':  torch.from_numpy(depth_2d),
        'time_enc':  torch.from_numpy(time_enc.astype(np.float32)),
        'T':         T,
        'dyn_1d_cols': DYN_1D,
        'dyn_2d_cols': DYN_2D,
    }
    del dyn_1d, dyn_2d
    gc.collect()
    return result

print('build_event_tensors() defined.')

build_event_tensors() defined.


## Cell 8 — Build static tensors & edge indices

In [11]:
def build_static_tensors(n1d, n2d, e1d, e2d, ei1d, ei2d, conn, cols, stats_static):
    FEAT_1D  = [cols['px_1d'], cols['py_1d'], cols['depth_1d'], cols['inv_el'],
                cols['surf_el'], cols['base_area'],
                'fill_ratio', 'degree_1d', 'n_2d_connections', 'catchment_area']
    FEAT_2D  = [cols['px_2d'], cols['py_2d'], cols['area_2d'], cols['rough_2d'],
                cols['min_el'], cols['elev_2d'],
                'aspect_sin', 'aspect_cos', 'area_log', 'flow_acc_log',
                'terrain_slope_proxy', 'fill_capacity', 'catchment_potential',
                'has_1d_connection', 'dist_to_1d', 'connected_1d_depth']
    FEAT_E1D = [cols['rpx_1d'], cols['rpy_1d'], cols['len_1d'],
                cols['diam_1d'], cols['slope_1d'], 'head_loss', 'capacity_proxy']
    FEAT_E2D = [cols['rpx_2d'], cols['rpy_2d'], cols['face_2d'],
                cols['len_2d'], cols['slope_2d'],
                'unit_conveyance', 'head_loss_2d', 'dx_norm', 'dy_norm']

    FEAT_1D  = [c for c in FEAT_1D  if c in n1d.columns]
    FEAT_2D  = [c for c in FEAT_2D  if c in n2d.columns]
    FEAT_E1D = [c for c in FEAT_E1D if c in e1d.columns]
    FEAT_E2D = [c for c in FEAT_E2D if c in e2d.columns]

    n1d_n = normalise_df(n1d.sort_values('node_idx'), stats_static['1d_node'], FEAT_1D)
    n2d_n = normalise_df(n2d.sort_values('node_idx'), stats_static['2d_node'], FEAT_2D)
    e1d_n = normalise_df(e1d, stats_static['1d_edge'], FEAT_E1D)
    e2d_n = normalise_df(e2d, stats_static['2d_edge'], FEAT_E2D)

    NORM_1D  = [f'{c}_norm' for c in FEAT_1D  if f'{c}_norm' in n1d_n.columns]
    NORM_2D  = [f'{c}_norm' for c in FEAT_2D  if f'{c}_norm' in n2d_n.columns]
    NORM_E1D = [f'{c}_norm' for c in FEAT_E1D if f'{c}_norm' in e1d_n.columns]
    NORM_E2D = [f'{c}_norm' for c in FEAT_E2D if f'{c}_norm' in e2d_n.columns]

    S1D  = torch.tensor(n1d_n[NORM_1D].fillna(0).values,  dtype=torch.float32)
    S2D  = torch.tensor(n2d_n[NORM_2D].fillna(0).values,  dtype=torch.float32)
    SE1D = torch.tensor(e1d_n[NORM_E1D].fillna(0).values, dtype=torch.float32)
    SE2D = torch.tensor(e2d_n[NORM_E2D].fillna(0).values, dtype=torch.float32)
    SE1D = torch.cat([SE1D, SE1D], dim=0)
    SE2D = torch.cat([SE2D, SE2D], dim=0)

    def undirected(frm, to):
        return torch.stack([
            torch.tensor(np.concatenate([frm, to]), dtype=torch.long),
            torch.tensor(np.concatenate([to, frm]), dtype=torch.long)
        ])

    def directed(frm, to):
        return torch.stack([
            torch.tensor(frm, dtype=torch.long),
            torch.tensor(to,  dtype=torch.long)
        ])

    EI_1D   = undirected(ei1d[cols['from_1d']].values, ei1d[cols['to_1d']].values)
    EI_2D   = undirected(ei2d[cols['from_2d']].values, ei2d[cols['to_2d']].values)
    EI_1D2D = directed(conn['node_1d'].values, conn['node_2d'].values)
    EI_2D1D = directed(conn['node_2d'].values, conn['node_1d'].values)
    COUP_E  = torch.zeros(EI_1D2D.shape[1], 4, dtype=torch.float32)

    INVERT_EL = torch.tensor(
        n1d.sort_values('node_idx').set_index('node_idx')[cols['inv_el']].values,
        dtype=torch.float32)
    MIN_EL = torch.tensor(
        n2d.sort_values('node_idx').set_index('node_idx')[cols['min_el']].values,
        dtype=torch.float32)

    return {
        'static_1d_node': S1D,  'static_2d_node': S2D,
        'static_1d_edge': SE1D, 'static_2d_edge': SE2D,
        'static_coup_edge': COUP_E,
        'edge_idx_1d': EI_1D,   'edge_idx_2d': EI_2D,
        'edge_idx_1d2d': EI_1D2D, 'edge_idx_2d1d': EI_2D1D,
        'invert_elevation': INVERT_EL, 'min_elevation': MIN_EL,
        'feat_cols': {'1d_node': NORM_1D, '2d_node': NORM_2D,
                      '1d_edge': NORM_E1D, '2d_edge': NORM_E2D},
    }

print('build_static_tensors() defined.')

build_static_tensors() defined.


## Cell 9 — Main loop: process both models

In [12]:
WARMUP = 10

for MODEL_ID in MODEL_IDS:
    print(f'\n{"="*60}')
    print(f'  Processing Model {MODEL_ID}')
    print(f'{"="*60}')

    model_dir = MODELS_ROOT / f'Model_{MODEL_ID}'
    out_dir   = OUT_ROOT / f'model_{MODEL_ID}'
    out_dir.mkdir(parents=True, exist_ok=True)

    # Step 1: Load & engineer statics
    print('Loading static files...')
    n1d, n2d, e1d, e2d, ei1d, ei2d, conn = load_static(model_dir)
    n1d, n2d, e1d, e2d, cols = engineer_static_features(
        n1d, n2d, e1d, e2d, ei1d, ei2d, conn)
    print(f'  N_1D={len(n1d)}, N_2D={len(n2d)}')

    # Step 2: Compute norm stats by streaming all train events
    print('Computing normalisation stats...')
    dyn_stats = collect_train_stats(model_dir, n1d, n2d, cols)

    # Static feature norm stats (from static tables, no dynamic data needed)
    FEAT_1D_S  = [cols['px_1d'], cols['py_1d'], cols['depth_1d'], cols['inv_el'],
                  cols['surf_el'], cols['base_area'],
                  'fill_ratio', 'degree_1d', 'n_2d_connections', 'catchment_area']
    FEAT_2D_S  = [cols['px_2d'], cols['py_2d'], cols['area_2d'], cols['rough_2d'],
                  cols['min_el'], cols['elev_2d'],
                  'aspect_sin', 'aspect_cos', 'area_log', 'flow_acc_log',
                  'terrain_slope_proxy', 'fill_capacity', 'catchment_potential',
                  'has_1d_connection', 'dist_to_1d', 'connected_1d_depth']
    FEAT_E1D_S = [cols['rpx_1d'], cols['rpy_1d'], cols['len_1d'],
                  cols['diam_1d'], cols['slope_1d'], 'head_loss', 'capacity_proxy']
    FEAT_E2D_S = [cols['rpx_2d'], cols['rpy_2d'], cols['face_2d'],
                  cols['len_2d'], cols['slope_2d'],
                  'unit_conveyance', 'head_loss_2d', 'dx_norm', 'dy_norm']

    stats_static = {
        '1d_node': compute_stats(n1d, FEAT_1D_S),
        '2d_node': compute_stats(n2d, FEAT_2D_S),
        '1d_edge': compute_stats(e1d, FEAT_E1D_S),
        '2d_edge': compute_stats(e2d, FEAT_E2D_S),
    }

    # Step 3: Build & save static tensors
    print('Building static tensors...')
    static_bundle = build_static_tensors(
        n1d, n2d, e1d, e2d, ei1d, ei2d, conn, cols, stats_static)
    torch.save(static_bundle, out_dir / 'static_tensors.pt')
    print(f'  Saved: 1D node {static_bundle["static_1d_node"].shape}, '
          f'2D node {static_bundle["static_2d_node"].shape}')

    # Step 4: Build & save train event tensors
    train_events = get_event_dirs(model_dir, 'train')
    (out_dir / 'train').mkdir(exist_ok=True)
    dyn_cols_saved = False
    print(f'Building {len(train_events)} train event tensors...')

    for ev_dir in tqdm(train_events, desc=f'  M{MODEL_ID} train'):
        ev_id = int(ev_dir.name.split('_')[1])
        tensors = build_event_tensors(ev_dir, n1d, n2d, conn, cols, dyn_stats)
        torch.save({
            'x_1d': tensors['x_1d'], 'x_2d': tensors['x_2d'],
            'delta_1d': tensors['delta_1d'], 'delta_2d': tensors['delta_2d'],
            'depth_1d': tensors['depth_1d'], 'depth_2d': tensors['depth_2d'],
            'time_enc': tensors['time_enc'], 'T': tensors['T'],
        }, out_dir / 'train' / f'event_{ev_id}.pt')
        if not dyn_cols_saved:
            with open(out_dir / 'dyn_col_names.json', 'w') as f:
                json.dump({'dyn_1d_cols': tensors['dyn_1d_cols'],
                           'dyn_2d_cols': tensors['dyn_2d_cols']}, f, indent=2)
            dyn_cols_saved = True
        del tensors
        gc.collect()

    # Step 5: Build & save test event tensors
    test_events = get_event_dirs(model_dir, 'test')
    (out_dir / 'test').mkdir(exist_ok=True)
    print(f'Building {len(test_events)} test event tensors...')

    for ev_dir in tqdm(test_events, desc=f'  M{MODEL_ID} test'):
        ev_id = int(ev_dir.name.split('_')[1])
        tensors = build_event_tensors(ev_dir, n1d, n2d, conn, cols, dyn_stats)
        torch.save({
            'x_1d_warmup':     tensors['x_1d'][:WARMUP],
            'x_2d_warmup':     tensors['x_2d'][:WARMUP],
            'x_1d_full':       tensors['x_1d'],
            'x_2d_full':       tensors['x_2d'],
            'depth_1d_warmup': tensors['depth_1d'][:WARMUP],
            'depth_2d_warmup': tensors['depth_2d'][:WARMUP],
            'time_enc':        tensors['time_enc'],
            'T':               tensors['T'],
        }, out_dir / 'test' / f'event_{ev_id}.pt')
        del tensors
        gc.collect()

    # Step 6: Save feature config
    with open(out_dir / 'dyn_col_names.json') as f:
        dyn_cols = json.load(f)

    feature_config = {
        'model_id': MODEL_ID, 'n_1d': len(n1d), 'n_2d': len(n2d),
        'f_1d_static':  static_bundle['static_1d_node'].shape[1],
        'f_2d_static':  static_bundle['static_2d_node'].shape[1],
        'f_1d_dyn':     len(dyn_cols['dyn_1d_cols']),
        'f_2d_dyn':     len(dyn_cols['dyn_2d_cols']),
        'f_1d_edge':    static_bundle['static_1d_edge'].shape[1],
        'f_2d_edge':    static_bundle['static_2d_edge'].shape[1],
        'n_train_events': len(train_events),
        'n_test_events':  len(test_events),
        'warmup_steps': WARMUP,
        'std_wl_1d':    dyn_stats['wl_1d']['std'],
        'std_wl_2d':    dyn_stats['wl_2d']['std'],
        'std_depth_1d': dyn_stats['depth_1d']['std'],
        'std_depth_2d': dyn_stats['depth_2d']['std'],
        'std_delta_1d': dyn_stats['delta_1d']['std'],
        'std_delta_2d': dyn_stats['delta_2d']['std'],
        'depth_mean_1d':dyn_stats['depth_1d']['mean'],
        'depth_mean_2d':dyn_stats['depth_2d']['mean'],
        'dyn_1d_cols':  dyn_cols['dyn_1d_cols'],
        'dyn_2d_cols':  dyn_cols['dyn_2d_cols'],
        'static_feat_cols': static_bundle['feat_cols'],
        'col_names':    cols,
    }
    with open(out_dir / 'feature_config.json', 'w') as f:
        json.dump(feature_config, f, indent=2, default=str)

    print(f'Model {MODEL_ID} complete.')
    print(f'  Train .pt files: {len(list((out_dir/"train").glob("*.pt")))}')
    print(f'  Test  .pt files: {len(list((out_dir/"test").glob("*.pt")))}')

    del n1d, n2d, e1d, e2d, ei1d, ei2d, conn, static_bundle
    gc.collect()

print('\nAll models processed.')


  Processing Model 1
Loading static files...
  N_1D=17, N_2D=3716
Computing normalisation stats...


  Scanning Model_1:   0%|          | 0/68 [00:00<?, ?it/s]

  WL std   -> 1D: 16.8778 ft,  2D: 14.3788 ft
  Depth std -> 1D: 0.4711 ft,  2D: 0.6327 ft
  Delta std -> 1D: 0.08014 ft,  2D: 0.01489 ft
Building static tensors...
  Saved: 1D node torch.Size([17, 10]), 2D node torch.Size([3716, 16])
Building 68 train event tensors...


  M1 train:   0%|          | 0/68 [00:00<?, ?it/s]

Building 29 test event tensors...


...

  WL std   -> 1D: 3.1918 ft,  2D: 2.7271 ft
  Depth std -> 1D: 3.2219 ft,  2D: 0.9233 ft
  Delta std -> 1D: 0.09626 ft,  2D: 0.01582 ft
Building static tensors...
  Saved: 1D node torch.Size([198, 10]), 2D node torch.Size([4299, 16])
Building 69 train event tensors...


  M2 train:   0%|          | 0/69 [00:00<?, ?it/s]

Building 30 test event tensors...


  M2 test:   0%|          | 0/30 [00:00<?, ?it/s]

Model 2 complete.
  Train .pt files: 69
  Test  .pt files: 30

All models processed.


## Cell 10 — Sanity check & summary

In [13]:
print('='*65)
print('  FULL DATASET PREPROCESSING - SANITY CHECK')
print('='*65)

for MODEL_ID in MODEL_IDS:
    out_dir = OUT_ROOT / f'model_{MODEL_ID}'
    with open(out_dir / 'feature_config.json') as f:
        fc = json.load(f)

    static      = torch.load(out_dir / 'static_tensors.pt', map_location='cpu')
    train_pts   = sorted((out_dir / 'train').glob('event_*.pt'))
    test_pts    = sorted((out_dir / 'test').glob('event_*.pt'))
    sample_train = torch.load(train_pts[0],  map_location='cpu')
    sample_test  = torch.load(test_pts[0],   map_location='cpu')

    checks = [
        ('Train events saved',      len(train_pts) == fc['n_train_events']),
        ('Test events saved',       len(test_pts)  == fc['n_test_events']),
        ('No NaN static 1D',        not static['static_1d_node'].isnan().any().item()),
        ('No NaN static 2D',        not static['static_2d_node'].isnan().any().item()),
        ('No NaN train x_1d',       not sample_train['x_1d'].isnan().any().item()),
        ('No NaN train x_2d',       not sample_train['x_2d'].isnan().any().item()),
        ('No NaN delta_1d target',  not sample_train['delta_1d'].isnan().any().item()),
        ('Depth_1d >= 0',           (sample_train['depth_1d'] >= -1e-4).all().item()),
        ('Depth_2d >= 0',           (sample_train['depth_2d'] >= -1e-4).all().item()),
        ('Test warmup = 10 steps',  sample_test['x_1d_warmup'].shape[0] == 10),
        ('Edge idx 1D in range',    static['edge_idx_1d'].max().item() < fc['n_1d']),
        ('Edge idx 2D in range',    static['edge_idx_2d'].max().item() < fc['n_2d']),
        ('STD_WL_1D > 0',           fc['std_wl_1d'] > 0),
        ('STD_WL_2D > 0',           fc['std_wl_2d'] > 0),
    ]

    print(f'\nModel {MODEL_ID}:')
    all_pass = True
    for name, passed in checks:
        print(f'  {"PASS" if passed else "FAIL"}  {name}')
        all_pass = all_pass and passed

    print(f'\n  N_1D={fc["n_1d"]}, N_2D={fc["n_2d"]}')
    print(f'  F_1D_static={fc["f_1d_static"]}, F_2D_static={fc["f_2d_static"]}')
    print(f'  F_1D_dyn={fc["f_1d_dyn"]}, F_2D_dyn={fc["f_2d_dyn"]}')
    print(f'  Train events={fc["n_train_events"]}, Test events={fc["n_test_events"]}')
    print(f'  STD_WL_1D={fc["std_wl_1d"]:.4f} ft, STD_WL_2D={fc["std_wl_2d"]:.4f} ft')
    print(f'  Sample train T={sample_train["T"]}, x_1d shape={tuple(sample_train["x_1d"].shape)}')
    print(f'  {"ALL CHECKS PASSED" if all_pass else "SOME CHECKS FAILED"}')

    del static, sample_train, sample_test
    gc.collect()

print('\nPreprocessing complete. Run urbanflood_training.ipynb next.')

  FULL DATASET PREPROCESSING - SANITY CHECK

Model 1:
  PASS  Train events saved
  PASS  Test events saved
  PASS  No NaN static 1D
  PASS  No NaN static 2D
  PASS  No NaN train x_1d
  FAIL  No NaN train x_2d
  PASS  No NaN delta_1d target
  PASS  Depth_1d >= 0
  FAIL  Depth_2d >= 0
  PASS  Test warmup = 10 steps
  PASS  Edge idx 1D in range
  PASS  Edge idx 2D in range
  PASS  STD_WL_1D > 0
  PASS  STD_WL_2D > 0

  N_1D=17, N_2D=3716
  F_1D_static=10, F_2D_static=16
  F_1D_dyn=9, F_2D_dyn=9
  Train events=68, Test events=29
  STD_WL_1D=16.8778 ft, STD_WL_2D=14.3788 ft
  Sample train T=94, x_1d shape=(94, 17, 9)
  SOME CHECKS FAILED

Model 2:
  PASS  Train events saved
  PASS  Test events saved
  PASS  No NaN static 1D
  PASS  No NaN static 2D
  PASS  No NaN train x_1d
  PASS  No NaN train x_2d
  PASS  No NaN delta_1d target
  PASS  Depth_1d >= 0
  PASS  Depth_2d >= 0
  PASS  Test warmup = 10 steps
  PASS  Edge idx 1D in range
  PASS  Edge idx 2D in range
  PASS  STD_WL_1D > 0
  PASS  

In [ ]:
import pandas as pd

for MODEL_ID in [1, 2]:
    out_dir   = OUT_ROOT / f'model_{MODEL_ID}'
    model_dir = MODELS_ROOT / f'Model_{MODEL_ID}'
    train_dir = model_dir / 'train'

    n1d = pd.read_csv(train_dir / '1d_nodes_static.csv')
    n1d.columns = n1d.columns.str.strip().str.lower()
    id_col = [c for c in n1d.columns if 'node' in c and ('idx' in c or 'id' in c)][0]
    node_ids_1d = sorted(n1d[id_col].tolist())

    n2d_path = train_dir / '2d_nodes_static.csv'
    if not n2d_path.exists():
        n2d_path = train_dir / '2d_nodes_index.csv'
    n2d = pd.read_csv(n2d_path)
    n2d.columns = n2d.columns.str.strip().str.lower()
    id_col2 = [c for c in n2d.columns if 'node' in c and ('idx' in c or 'id' in c)][0]
    node_ids_2d = sorted(n2d[id_col2].tolist())

    node_ids = {'node_ids_1d': node_ids_1d, 'node_ids_2d': node_ids_2d}
    with open(out_dir / 'node_ids.json', 'w') as f:
        json.dump(node_ids, f)

    print(f'Model {MODEL_ID}: saved {len(node_ids_1d)} 1D node IDs, {len(node_ids_2d)} 2D node IDs')

In [14]:
import torch, json, gc
from pathlib import Path
import numpy as np

PREP_ROOT = Path('/kaggle/working/preprocessed')

for MODEL_ID in [1, 2]:
    print(f'\n{"="*50}')
    print(f'Diagnosing Model {MODEL_ID}')
    print(f'{"="*50}')

    out_dir = PREP_ROOT / f'model_{MODEL_ID}'

    # Load a sample train event
    train_pts = sorted((out_dir / 'train').glob('event_*.pt'),
                       key=lambda p: int(p.stem.split('_')[1]))
    ev = torch.load(train_pts[0], map_location='cpu')

    x_2d     = ev['x_2d']      # (T, N_2D, F_dyn)
    depth_2d = ev['depth_2d']  # (T, N_2D)

    # ── Check 1: NaN in x_2d ──────────────────────────────────────────────
    nan_mask = torch.isnan(x_2d)
    print(f'\nx_2d NaN count: {nan_mask.sum().item()}')
    if nan_mask.any():
        # Which feature columns have NaN?
        nan_per_feat = nan_mask.any(dim=0).any(dim=0)  # (F_dyn,)
        print(f'  NaN in feature cols: {nan_per_feat.nonzero(as_tuple=True)[0].tolist()}')
        # Which nodes have NaN?
        nan_per_node = nan_mask.any(dim=0).any(dim=1)  # (N_2D,)
        nan_nodes = nan_per_node.nonzero(as_tuple=True)[0]
        print(f'  Nodes with NaN: {len(nan_nodes)} / {x_2d.shape[1]}')
        print(f'  First 10 nan node indices: {nan_nodes[:10].tolist()}')

    # ── Check 2: negative depth_2d ────────────────────────────────────────
    neg_mask = depth_2d < -1e-4
    print(f'\ndepth_2d negative count: {neg_mask.sum().item()}')
    if neg_mask.any():
        neg_per_node = neg_mask.any(dim=0)
        neg_nodes = neg_per_node.nonzero(as_tuple=True)[0]
        print(f'  Nodes with negative depth: {len(neg_nodes)} / {depth_2d.shape[1]}')
        print(f'  Min depth_2d value: {depth_2d.min().item():.6f}')
        print(f'  First 5 neg node indices: {neg_nodes[:5].tolist()}')
        # Are these the same nodes as NaN nodes?
        if nan_mask.any():
            overlap = set(nan_nodes.tolist()) & set(neg_nodes.tolist())
            print(f'  Overlap with NaN nodes: {len(overlap)}')

    del ev
    gc.collect()


Diagnosing Model 1

x_2d NaN count: 1128
  NaN in feature cols: [0]
  Nodes with NaN: 12 / 3716
  First 10 nan node indices: [3704, 3705, 3706, 3707, 3708, 3709, 3710, 3711, 3712, 3713]

depth_2d negative count: 0

Diagnosing Model 2

x_2d NaN count: 0

depth_2d negative count: 0


In [17]:
for MODEL_ID in [1, 2]:
    print(f'\nFixing Model {MODEL_ID}...')
    out_dir = PREP_ROOT / f'model_{MODEL_ID}'

    for split in ['train', 'test']:
        pts = sorted((out_dir / split).glob('event_*.pt'),
                     key=lambda p: int(p.stem.split('_')[1]))
        n_modified = 0

        for pt_path in pts:
            ev = torch.load(pt_path, map_location='cpu')
            modified = False

            # Train files store x_1d/x_2d, test files store warmup/full variants.
            for k in ['x_2d', 'x_2d_warmup', 'x_2d_full']:
                if k in ev and torch.isnan(ev[k]).any():
                    ev[k] = torch.nan_to_num(ev[k], nan=0.0)
                    modified = True

            for k in ['x_1d', 'x_1d_warmup', 'x_1d_full']:
                if k in ev and torch.isnan(ev[k]).any():
                    ev[k] = torch.nan_to_num(ev[k], nan=0.0)
                    modified = True

            # Depth keys differ between train and test bundles.
            for k in ['depth_2d', 'depth_2d_warmup']:
                if k in ev and (ev[k] < 0).any():
                    ev[k] = ev[k].clamp(min=0.0)
                    modified = True

            for k in ['depth_1d', 'depth_1d_warmup']:
                if k in ev and (ev[k] < 0).any():
                    ev[k] = ev[k].clamp(min=0.0)
                    modified = True

            # Delta targets exist only for train events.
            for k in ['delta_1d', 'delta_2d']:
                if k in ev:
                    fixed = torch.nan_to_num(ev[k], nan=0.0)
                    if not torch.equal(fixed, ev[k]):
                        ev[k] = fixed
                        modified = True

            if modified:
                torch.save(ev, pt_path)
                n_modified += 1

            del ev
            gc.collect()

        print(f'  {split}: {n_modified} / {len(pts)} events updated')

    print(f'Model {MODEL_ID} done.')


Fixing Model 1...
  train: 0 / 68 events updated
  test: 29 / 29 events updated
Model 1 done.

Fixing Model 2...
  train: 0 / 69 events updated
  test: 30 / 30 events updated
Model 2 done.
